# MILP Allocator

In [2]:
import os

from copy import deepcopy

from sim_types import GPUType
from sim_types import Objective

from constants import DEFAULT_WORKFLOW_CONFIG

from data_loading import load_latency_data
from data_loading import load_power_data

from policies import STREAMWISE_POLICY
from policies import STREAMWISE_MILP_POLICY

from utils import to_models_df

from milp import MILPAllocator

In [3]:
workflow = DEFAULT_WORKFLOW_CONFIG

data_dir = "data/"
latency_data = load_latency_data(data_dir=data_dir)
power_data = load_power_data(data_dir=data_dir)

## Single setting

In [4]:
policy_ttff_cost = deepcopy(STREAMWISE_MILP_POLICY)
policy_ttff_cost.objective = Objective.TTFF_COST

policy_ttff = deepcopy(STREAMWISE_MILP_POLICY)
policy_ttff.objective = Objective.TTFF

policy_cost = deepcopy(STREAMWISE_MILP_POLICY)
policy_cost.objective = Objective.COST

policy_energy = deepcopy(STREAMWISE_MILP_POLICY)
policy_energy.objective = Objective.ENERGY


In [ ]:
# Find fastest TTFF and cheapest for each GPU type
allocator_ttff = MILPAllocator(
    workflow=workflow,
    latency_data=latency_data,
    power_data=power_data,
    policy=policy_ttff,
)
allocator_cost = MILPAllocator(
    workflow=workflow,
    latency_data=latency_data,
    power_data=power_data,
    policy=policy_cost,
)
for gpu_type in GPUType:
    # Find fastest TTFF
    result_ttff = allocator_ttff.allocate(
        num_gpus={gpu_type: 512},
        time_limit=2*60,
        save_solution_path=f"data/solution_ttff_{gpu_type.value}.json",
    )
    if result_ttff:
        with open("data/results.csv", "a") as file:
            file.write(result_ttff.to_csv() + "\n")
        print(result_ttff.to_csv())

        # Find cheapest cost with the TTFF
        result_cost = allocator_cost.allocate(
            num_gpus={gpu_type: 512},
            time_limit=2*60,
            max_ttff=result_ttff.ttff_s,
            warm_start_path=f"data/solution_ttff_{gpu_type.value}.json",
            save_solution_path=f"data/solution_cost_{gpu_type.value}.json",
        )
        if result_cost:
            with open("data/results.csv", "a") as file:
                file.write(result_cost.to_csv() + "\n")
            print(result_cost.to_csv())

In [ ]:
to_models_df(result_cost.models)

In [ ]:
# Run the whole sweep of TTFF
allocator = MILPAllocator(
    workflow=workflow,
    latency_data=latency_data,
    power_data=power_data,
    policy=policy_cost,
)
for gpu_type in GPUType:
    prev_ttff = None
    for ttff in [
        50, 60, 100, 2*60, 3*60, 5*60, 8*60, 10*60, 15*60, 20*60, 30*60, 45*60, 60*60,
        2*60*60, 3*60*60, 4*60*60, 5*60*60, 6*60*60, 7*60*60, 8*60*60,
    ]:
        file_solution_new = f"data/milp_solution_{gpu_type}_{ttff}.json"
        file_solution_old = f"data/milp_solution_{gpu_type}_{prev_ttff}.json"
        if not os.path.exists(file_solution_old):
            file_solution_old = None

        result = allocator.allocate(
            num_gpus={gpu_type: 256},
            time_limit=2*60,
            max_ttff=ttff,
            save_solution_path=file_solution_new,
            warm_start_path=file_solution_old,
        )
        if result:
            with open("data/results.csv", "a") as file:
                file.write(result.to_csv() + "\n")
            print(result.to_csv())
        prev_ttff = ttff

## Exploration
### Single GPU type

In [ ]:
for gpu_type in GPUType:
    for num in [8, 16, 24, 32, 48, 64, 96, 128, 192, 256]:
        allocator = MILPAllocator(
            workflow=workflow,
            latency_data=latency_data,
            power_data=power_data,
            policy=policy_ttff_cost,
        )
        result = allocator.allocate(
            num_gpus={gpu_type: num},
            # verbose=True,
            time_limit=5*60,  # 5 minutes
            # force_num_gpus=True,
        )
        if result:
            print(result.to_csv())

In [ ]:
# Fastest TTFF and cheapest overall
results_fastest: dict[GPUType, float] = {}
results_cheapest: dict[GPUType, float] = {}
for gpu_type in GPUType:
    for policy in [policy_ttff, policy_cost]:
        allocator = MILPAllocator(
            workflow=workflow,
            latency_data=latency_data,
            power_data=power_data,
            policy=policy,
        )
        result = allocator.allocate(
            num_gpus={gpu_type: 256},
            time_limit=5*60,  # 5 minutes
            # force_num_gpus=True,
        )
        if result:
            print(result.to_csv())
            if policy.objective == Objective.TTFF:
                results_fastest[gpu_type] = result.ttff_s
            elif policy.objective == Objective.COST:
                results_cheapest[gpu_type] = result.cost

In [ ]:
# Cheapest for the fastest TTFF
for gpu_type in GPUType:
    allocator = MILPAllocator(
        workflow=workflow,
        latency_data=latency_data,
        power_data=power_data,
        policy=policy_cost,
    )
    result = allocator.allocate(
        num_gpus={gpu_type: 256},
        max_ttff=results_fastest[gpu_type] * 1.01,  # 1% slower than the fastest
        time_limit=5*60,  # 5 minutes
        # force_num_gpus=True,
    )
    if result:
        print(result.to_csv())

## GPU Pair

In [ ]:
for gpu_type1, gpu_type2 in [
    (GPUType.A100, GPUType.H100),
    (GPUType.A100, GPUType.H200),
    (GPUType.A100, GPUType.GB200),
    (GPUType.H100, GPUType.H200),
    (GPUType.H100, GPUType.GB200),
    (GPUType.H200, GPUType.GB200),
]:
    for num in [8, 16, 24, 32, 48, 64, 96, 128, 192, 256]:
        allocator = MILPAllocator(
            workflow=workflow,
            latency_data=latency_data,
            power_data=power_data,
            policy=policy_ttff_cost,
        )
        result = allocator.allocate(
            num_gpus={
                gpu_type1: num,
                gpu_type2: num,
            },
            time_limit=5*60,  # 5 minutes
            # force_num_gpus=True,
        )
        if result:
            print(result.to_csv())

## Compare to greedy heuristic

In [5]:
from greedy import GreedyAllocator
from evaluator import evaluate_model_allocation

greedy_allocator = GreedyAllocator(
    workflow=workflow,
    latency_data=latency_data,
    power_data=power_data,
    policy=STREAMWISE_POLICY,
)
result_greedy = greedy_allocator.allocate(
    num_gpus={
        GPUType.A100: 256,
        GPUType.H200: 64,
    },
    verbose=False,
)

# Validate values by running through the evaluator
result_validated = evaluate_model_allocation(
    models=result_greedy.models,
    num_gpus={
        GPUType.A100: 256,
        GPUType.H200: 64,
    },
    workflow=workflow,
    latency_data=latency_data,
    power_data=power_data,
    policy=STREAMWISE_POLICY,
)

print(result_validated)


to_models_df(result_validated.models)

Time:301.88 s TTFF:21.38 s Cost:$45.62 TTFF*Cost:975.42 Energy:4.96 kWh GPUS: 256xA100+64xH200


Devices  Replicas  Work  #GPUs  Time (s)  TTFF (s)  \
A100  gemma           8         1     0      8      8.57      1.48   
      flux           16         1     0     16      0.95      0.95   
      hf              2         6     0     12     29.27      2.03   
      hf              1        29     0     29     29.02      2.97   
      hf_vae          1        20     0     20     11.75      1.98   
      ft              4        18     0     72    190.44     58.10   
      ft              2        12     0     24    190.37     78.77   
      upscaler        4         8     0     32     34.75      3.92   
      upscaler        8         3     0     24     34.75      2.00   
      upscaler        2         6     0     12     34.76      7.76   
      upscaler        1         6     0      6     34.77     15.44   
      others          1         1     0      1     25.80      0.60   
H200  hf              2         4     0      8     29.21      0.90   
      hf_vae          1         4     0      4     11.75      0.86   
      ft              2        13     0     26    190.76     34.44   
      ft             24         1     0     24    189.73     14.59   
      upscaler        2         1     0      2     34.75      3.82   
TOTAL                81       134     0    320    301.87     21.38   

                Energy (kWh)  Cost ($)  
A100  gemma             0.24      0.72  
      flux              0.09      1.44  
      hf                0.07      1.08  
      hf                0.17      2.60  
      hf_vae            0.11      1.79  
      ft                1.60      6.46  
      ft                0.54      2.15  
      upscaler          0.23      2.87  
      upscaler          0.17      2.15  
      upscaler          0.08      1.08  
      upscaler          0.04      0.54  
      others            0.00      0.09  
H200  hf                0.06      2.83  
      hf_vae            0.03      1.42  
      ft                0.80      9.20  
      ft                0.71      8.49  
      upscaler          0.02      0.71  
TOTAL                   4.96     45.62

# Plot

In [ ]:
import sys
sys.path.append("../simulator")

from plot_utils import _get_time_ticklabels
from utils import get_pareto_frontier_paper

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# these are ints df_milp[["num_a100", "num_h100", "num_h200", "num_gb200"]]
filename = "../paper/data/provisioning_streamwise_milp.csv"
df_milp = pd.read_csv(filename, comment="#")
df_milp["ttff_s*cost"] = (df_milp["ttff_s"] * df_milp["cost"]).round(2)
df_milp["energy_kwh"] = (df_milp["energy"] / 1000 / 3600).round(2)

In [ ]:
df_milp.sort_values(["ttff_s", "cost"]).head(20).reset_index(drop=True)
# df_milp.sort_values(["ttff_s*cost"]).head(20).reset_index(drop=True)

In [ ]:
PAPER_FIG_SIZE = (5.5, 2)
PAPER_FIG_SIZE_TALL = (5.5, 2.5)

fig, ax = plt.subplots(figsize=PAPER_FIG_SIZE_TALL)

plt.scatter(
    x=df_milp["ttff_s"],
    y=df_milp["cost"],
    zorder=1
)

pareto_front = get_pareto_frontier_paper(
    df_milp[["ttff_s", "cost"]].to_numpy(),
    max_x=10 * 60 * 60,
)
plt.plot(
    pareto_front[:, 0],
    pareto_front[:, 1],
    linewidth=3,
    label="Frontier",
    zorder=0
)
plt.xscale("log")
ticks, tick_labels = _get_time_ticklabels()
plt.xticks(ticks, tick_labels)
plt.xlim(8, 5 * 60 * 60)
plt.ylim(0, 220)

ax.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

plt.tight_layout(pad=0)
plt.show()